# Multi-turn main run v5 (topic ladders)

12 topics (6 old + 6 meta) × 2 models × 2 conversation orders × pressure/neutral = 96 histories. The pressure text is the frozen ladder file `docs/main_run_ladders.json`; analysis follows `docs/readout_rule.md`.

Use a GPU runtime and check the actual CU/hour. Run cells in order, not Run all. **Run the smoke (step 4) first and review its parse rates before step 5.** Download results before disconnecting. The bundle is read from `MyDrive/multiturn-v5/multiturn-v5.zip` (build it with `python3 scripts/build_multiturn_bundle.py` from a clean checkout); without it, step 1 asks for an upload.

In [ ]:
# The bundle is read from Google Drive (MyDrive/multiturn-v5/multiturn-v5.zip); if it is not there, upload it.
from google.colab import drive, files
import os, zipfile, json, hashlib
from pathlib import Path
drive.mount("/content/drive")
BUNDLE = Path("/content/drive/MyDrive/multiturn-v5/multiturn-v5.zip")
if BUNDLE.exists():
    zip_path = str(BUNDLE)
else:
    uploaded = files.upload()
    # Colab renames a repeat upload ("multiturn-v5 (1).zip"); accept that, but only one bundle.
    names = [n for n in uploaded if n.startswith("multiturn-v5") and n.endswith(".zip")]
    assert len(names) == 1, f"Upload exactly one multiturn-v5 zip; got {list(uploaded)}"
    zip_path = names[0]
with zipfile.ZipFile(zip_path) as z:
    z.extractall("/content/multiturn-v5")
os.chdir("/content/multiturn-v5")
print("Bundle:", zip_path, json.load(open("provenance/build.json")))
print("Ladders sha256:", hashlib.sha256(open("docs/main_run_ladders.json", "rb").read()).hexdigest())


In [ ]:
%pip -q install "transformers==5.22.1" accelerate numpy huggingface_hub


In [ ]:
from google.colab import userdata
from huggingface_hub import login, HfApi
login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)
import torch
from pathlib import Path
assert torch.cuda.is_available(), "Select a GPU runtime"
print("GPU:", torch.cuda.get_device_name(0))
MODELS = {"llama": "meta-llama/Llama-3.1-8B-Instruct", "qwen": "Qwen/Qwen2.5-7B-Instruct"}
# Same pinned revisions as pilots v3/v4.
REVISIONS = {"llama": "0e9e39f249a16976918f6564b8830bc894c89659",
             "qwen": "a09a35458c702b33eeacc393d103063234e8bc28"}
for name, repo in MODELS.items():
    latest = HfApi().model_info(repo).sha
    print(name, "pinned", REVISIONS[name], "| latest on Hub", latest, "(same)" if latest == REVISIONS[name] else "(differs; pinned is used)")
TOPICS = [t["topic"] for t in json.load(open("docs/main_run_ladders.json"))["topics"]]
print(len(TOPICS), "topics:", TOPICS)


In [ ]:
# Keep results on Google Drive so a disconnect does not lose them; reruns skip finished histories.
from google.colab import drive
import subprocess, sys, time
drive.mount("/content/drive")
RESULTS = Path("/content/drive/MyDrive/multiturn-v5-results")
RESULTS.mkdir(parents=True, exist_ok=True)
link = Path("/content/multiturn-v5/results")
if link.is_symlink() or not link.exists():
    if link.is_symlink(): link.unlink()
    link.symlink_to(RESULTS)
else:
    raise RuntimeError("results/ already exists as a folder; move it to Drive by hand first")
print("results ->", RESULTS, "| existing files:", sum(1 for _ in RESULTS.rglob("*.json")))

def stream(cmd):
    """Run a child process and show its output in this cell (plain subprocess.run output does not
    appear in Colab). Returns the exit code; a non-zero code is printed, never swallowed."""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    code = p.wait()
    if code:
        print(f"*** exit code {code}: see the lines above ***", flush=True)
    return code

def run(cmd):
    """One history in its own process; outputs are saved after every turn."""
    return stream([sys.executable, "-u", "scripts/new_dialogue_v5.py", *cmd])


## Step 4: opening smoke (parse check, ~10–20 min)

Each model is loaded once and generates all 24 openings (12 topics × 2 orders) with exactly the main run's prompt and greedy settings, so these are the main run's openings. It checks the wording-only parse. An unparsed opening means that pressure history will not run; earlier screens suggest `ai_decline_lawful` is the one at risk. It does not estimate compute; step 4b does. Lines marked `CHECK` need a look before step 5.

In [ ]:
smoke_failed = []
for name, repo in MODELS.items():
    out = Path(f"results/smoke/openings_{name}.json")
    out.parent.mkdir(parents=True, exist_ok=True)
    print("==", name, flush=True)
    code = stream([sys.executable, "-u", "scripts/opening_smoke_v5.py", "--model", repo, "--revision", REVISIONS[name],
                   "--device", "cuda", "--out", str(out)])
    if code or not out.exists():
        smoke_failed.append(name)
print("FAILED:" if smoke_failed else "Both models done.", smoke_failed or "", "| files:", sorted(p.name for p in Path("results/smoke").glob("*.json")))


## Step 4b: timing (4 full histories, then a projection)

Runs, with `MAIN_DECODE` (ngram20, adopted in step 4c), the first topic's order-1 pressure and neutral histories for each model. They are real main-run histories, saved where step 5 expects them, so step 5 skips them. Set `CU_PER_HOUR` to the rate Colab shows. The projection is rough: topics differ in reply length, and one topic per model is a small sample. The 4 histories run here before the decision (no flag) are moved to `results/main_noflag/` and rerun.

In [ ]:
CU_PER_HOUR = None  # e.g. 7.5, from the Colab resources panel
TIMING_TOPIC = TOPICS[0]
# Adopted after step 4c (docs/run_v5.md): exposure and release turns use a 20-token no-repeat block,
# through a counting wrapper that records the steps where it changed the greedy token.
MAIN_DECODE = ["--no-repeat-ngram", "20"]
MAIN_DECODE_EXTRA = {"no_repeat_ngram_size": 20}   # what new_dialogue_v5.py records for MAIN_DECODE
def main_out(name, topic, order, condition):
    return Path(f"results/main/{name}_{topic}_o{order}_{condition}.json")
def run_history(name, topic, order, condition):
    out = main_out(name, topic, order, condition)
    if out.exists():
        prev = json.loads(out.read_text())
        if prev["args"].get("revision") != REVISIONS[name]:
            raise RuntimeError(f"Revision mismatch at {out}; inspect before rerunning")
        if prev.get("decode_extra", {}) != MAIN_DECODE_EXTRA:
            # Run before the decode setting was fixed (the first 4b histories): keep it aside, rerun.
            aside = out.parent.parent / "main_noflag" / out.name
            aside.parent.mkdir(parents=True, exist_ok=True)
            if aside.exists():
                raise RuntimeError(f"{aside} already exists; inspect before rerunning")
            out.rename(aside); print("Moved pre-decision output", out.name, "->", aside, flush=True)
            prev = None
    if out.exists():
        if prev.get("status") in ("complete", "truncated_history", "opening_unparsed"):
            return prev   # finished, or stopped by rule: never rerun
        # No status: interrupted mid-history. Greedy decoding reruns it identically; keep the partial file.
        aside = out.with_name(out.stem + f".interrupted-{int(time.time())}.json.part")
        out.rename(aside); print("Moved interrupted", out.name, "->", aside.name, flush=True)
    print("Running", out.stem, flush=True)
    t0 = time.time()
    code = run(["--model", MODELS[name], "--revision", REVISIONS[name], "--device", "cuda", "--topic", topic,
                "--order", str(order), "--condition", condition, "--out", str(out), *MAIN_DECODE])
    r = json.loads(out.read_text()) if out.exists() else {}
    r["_wall_seconds"] = time.time() - t0
    hit = [i for i, t in enumerate(r.get("turns", [])) if t.get("ngram_changed_steps")]
    print(out.stem, "exit", code, "status", r.get("status", "no_output"), f"{r['_wall_seconds']:.0f}s wall",
          "| n-gram block changed turns", hit or "none", flush=True)
    return r
per_model = {}
for name in MODELS:
    rs = [run_history(name, TIMING_TOPIC, 1, c) for c in ("pressure", "neutral")]
    # Only finished histories say how long a history takes; a stopped one is far shorter.
    secs = [r.get("_wall_seconds") or r.get("elapsed_seconds") for r in rs if r.get("status") == "complete"]
    if len(secs) < 2:
        print(f"WARNING {name}: only {len(secs)} of 2 timing histories completed; set TIMING_TOPIC to another topic")
    if secs:
        per_model[name] = sum(secs) / len(secs)
        print(f"{name}: {[round(s) for s in secs]} s per completed history")
total_h = sum(48 * s for s in per_model.values()) / 3600  # models without a timing are left out
print(f"Projected main run: about {total_h:.1f} GPU hours for 96 histories (both models).")
if CU_PER_HOUR:
    print(f"At {CU_PER_HOUR} CU/hour: about {total_h * CU_PER_HOUR:.0f} CU.")


## Step 4c: anti-repetition test (done 2026-11-14; decision: ngram20)

The Llama pressure history `four_day_week` o1 looped at exposure turn 5 and was truncated. This cell reran only that history under three settings into `results/decode_test/` (not `results/main/`). Result: `none` reproduced the loop exactly; `ngram20` kept turns 1–2 identical and the same stances, and turn 5 ended with EOS; `rep1.1` changed the stance from turn 1 on (it held the opening through exposure 4). **Adopted: `--no-repeat-ngram 20` for exposure and release turns, both models and conditions** (`MAIN_DECODE` in step 4b). Details in `docs/run_v5.md`. Kept for the record; no need to rerun.

In [ ]:
VARIANTS = {"none": [], "ngram20": ["--no-repeat-ngram", "20"], "rep1.1": ["--repetition-penalty", "1.1"]}
for label, flags in VARIANTS.items():
    out = Path(f"results/decode_test/llama_four_day_week_o1_pressure_{label}.json")
    if out.exists():
        print("exists:", out.name); continue
    out.parent.mkdir(parents=True, exist_ok=True)
    print("==", label, flush=True)
    run(["--model", MODELS["llama"], "--revision", REVISIONS["llama"], "--device", "cuda", "--topic", "four_day_week",
         "--order", "1", "--condition", "pressure", "--out", str(out), *flags])
for label in VARIANTS:
    out = Path(f"results/decode_test/llama_four_day_week_o1_pressure_{label}.json")
    if not out.exists():
        print(label, "no output"); continue
    r = json.loads(out.read_text())
    print(f"{label:8} {r.get('status'):18} tokens per turn {[t['tokens'] for t in r['turns']]} "
          f"finish {''.join(t['finish'][0] for t in r['turns'])}")


## Step 5: main run

96 histories (48 per model); the 4 from step 4b are skipped. Results go to Google Drive, so after a disconnect: reconnect, rerun the setup cells (through the Drive cell) and the step 4b cell's definitions, then this cell. Finished histories are skipped and an interrupted one is moved aside and rerun (greedy decoding reproduces it). Histories stopped by rule (`truncated_history`, `opening_unparsed`) are kept and never rerun. Set `MODEL_ORDER = ["llama"]` to do one model per session. Every history uses `MAIN_DECODE` from step 4b; an output in `results/main/` with another decode setting is moved to `results/main_noflag/` and rerun.

In [ ]:
results = {}
MODEL_ORDER = list(MODELS)  # e.g. ["llama"] to run one model per session
for name in MODEL_ORDER:
    for topic in TOPICS:
        for order in (1, 2):
            for condition in ("pressure", "neutral"):
                results[f"{name}_{topic}_o{order}_{condition}"] = run_history(name, topic, order, condition).get("status", "no_output")
from collections import Counter
print(Counter(results.values()))
print("Any status other than 'complete' must be inspected before use.")


In [ ]:
import subprocess, sys, shutil
# Readout-rule analysis (docs/readout_rule.md) -> results/analysis_v5/summary.md
subprocess.run([sys.executable, "scripts/analyze_v5.py", "results/main", "--out", "results/analysis_v5"])
print(open("results/analysis_v5/summary.md").read()[:4000])
from google.colab import files
shutil.make_archive("/content/multiturn-v5-results", "zip", "/content/multiturn-v5", "results")
files.download("/content/multiturn-v5-results.zip")


Download and verify results before disconnecting and deleting the GPU runtime. The script saves after each turn and readout but does not resume a partly completed history.